[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flizamar/entrenamiento-ioai-web/blob/gh-pages/plantillas/template.ipynb)

# Plantilla de trabajo · IOAI

**Tarea**: · **Ronda**: · **Métrica**: · **Límite de ejecución**:

**Baseline local**: · **Split**: · **Referencia oficial y su split**:

**Archivos exigidos**: · **Modelos permitidos**: · **Datos/etiquetas autorizados**:

El plan, en este orden y sin saltarse pasos:

1. **Mirar los datos** antes de modelar.
2. **Baseline**: el primer envío, que asegura el formato de salida.
3. **Modelo**: una mejora barata primero, profundizar solo si sobra tiempo.
4. **Evaluar** con la métrica oficial, sobre una validación honesta.
5. **Restart & Run all** antes de enviar.

Y la regla que ordena todo lo demás: en la IOAI **empatar el baseline vale
cero**, así que lo primero es despegarse de él — no resolver la tarea.

## 0. Preparación y ensayo offline

Primero instala el entorno de estudio y prepara los datos y pesos según el
contrato de la tarea. Ver `referencia/Entorno de estudio.md`. Esa preparación se
hace antes de activar el modo offline. Registra fuente, versión y archivos.

La plantilla arranca en aprendizaje. Activa `SIMULAR_COMPETENCIA` cuando hayas
comprobado los activos y quieras ensayar una ejecución sin descargas. Las
banderas de Hugging Face pueden leer archivos de caché mediante un ID; no
comprueban por sí mismas qué modelo está permitido. Si el enunciado exige rutas
locales, úsalo así.

Estas ayudas bloquean algunos comandos de instalación y miden bloques. No
constituyen el aislamiento ni el límite estricto del juez. Ensaya también la
carga, entrenamiento, inferencia y escritura desde un kernel nuevo.

> Es código **generado** desde la receta «Modo competencia» del vault, que se
> ejecuta contra sus pruebas en cada publicación. No lo edites acá: edita la
> receta y regenera con `python3 herramientas/generar_plantilla.py`.

In [ ]:
import os
import re
import time

PATRON_INSTALACION = re.compile(
    r"^\s*[!%]\s*(pip3?|conda|mamba|uv|apt|apt-get)\b.*\binstall\b", re.I)


def rechazar_instalacion(codigo):
    """Si el código intenta instalar un paquete, devuelve el motivo; si no, None.

    En la IOAI **está prohibido instalar paquetes**: lo que hay en la máquina es
    lo que hay. Entrenar donde `!pip install` funciona construye justo el
    reflejo que falla el día de la prueba.
    """
    for linea in codigo.splitlines():
        if PATRON_INSTALACION.match(linea):
            return ("prohibido instalar paquetes: %r. En la competencia no se "
                    "puede, así que resuélvelo con lo que ya está instalado."
                    % linea.strip())
    return None


def activar_offline():
    """Pone el ecosistema HuggingFace en modo offline y devuelve qué cambió.

    Actívalo ANTES de importar las librerías de HuggingFace. Evita sus
    consultas HTTP; un id del Hub puede funcionar si está completo en caché.
    No comprueba qué modelos permite la tarea ni desconecta toda la red.
    Prepara primero los archivos locales y usa las rutas del enunciado.
    """
    banderas = {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
                "HF_DATASETS_OFFLINE": "1"}
    os.environ.update(banderas)
    return banderas


def solo_de_train(ruta):
    """Guarda opcional para ejercicios que SOLO permiten etiquetas de train.

    Comprueba el nombre de la ruta, no su contenido ni sus permisos. Adapta
    esta política al enunciado: tareas adversariales como Double Agent Dilemma
    sí entregan y permiten etiquetas de test para construir el ataque.
    """
    partes = re.split(r"[/\\]", str(ruta))
    if not any(p.lower().startswith("train") for p in partes):
        raise ValueError(
            "esta guarda exige una ruta de train/, y %r no lo es. Comprueba "
            "si esa política corresponde al enunciado antes de usarla."
            % str(ruta))
    return ruta


class Presupuesto:
    """Cronómetro del límite de ejecución de una tarea.

        with Presupuesto(minutos=10, nombre="Ghost of the Machine") as p:
            entrenar(...)
            print("%.0f s de margen" % p.restante())

    Al salir informa cuánto se usó; si se pasó del límite levanta TimeoutError,
    y si pasó del 60% avisa. Ese margen no es paranoia: el límite cubre también
    la re-ejecución del notebook sobre los datos ocultos, en una máquina que no
    es la tuya.
    """

    def __init__(self, minutos=10.0, nombre="la tarea", margen=0.6):
        self.limite = minutos * 60.0
        self.nombre = nombre
        self.margen = margen
        self.inicio = None
        self.usado = None

    def __enter__(self):
        self.inicio = time.perf_counter()
        return self

    def transcurrido(self):
        return time.perf_counter() - self.inicio

    def restante(self):
        return self.limite - self.transcurrido()

    def __exit__(self, tipo, valor, traza):
        self.usado = self.transcurrido()
        pct = (100.0 * self.usado / self.limite) if self.limite else 0.0
        print("⏱️  %s: %.1f s de %.0f s (%.0f%% del límite)"
              % (self.nombre, self.usado, self.limite, pct))
        if tipo is not None:
            return False                    # ya hay un error: no lo tapamos
        if self.usado > self.limite:
            raise TimeoutError(
                "%s se pasó del límite: %.1f s de %.0f s permitidos."
                % (self.nombre, self.usado, self.limite))
        if self.usado > self.limite * self.margen:
            print("⚠️  usaste más del %.0f%% del límite. El juez re-ejecuta el "
                  "notebook sobre los datos ocultos: deja margen."
                  % (self.margen * 100))
        return False


def modo_competencia(offline=True, bloquear_instalaciones=True):
    """Activa ayudas de simulacro y devuelve cuáles pudo activar.

    Debe ejecutarse tras preparar activos y antes de importar HuggingFace.
    No crea un sandbox ni impone todas las reglas de la plataforma oficial.
    """
    activado = {}
    if offline:
        activado["offline"] = activar_offline()
    if bloquear_instalaciones:
        try:
            ip = get_ipython()              # noqa: F821  (existe en Jupyter)
        except NameError:
            ip = None
        if ip is None:
            activado["instalaciones"] = "sin bloquear (fuera de IPython)"
        else:
            def guardia_instalaciones(lineas):
                motivo = rechazar_instalacion("".join(lineas))
                if motivo:
                    raise RuntimeError(motivo)
                return lineas

            previos = [t for t in ip.input_transformers_cleanup
                       if getattr(t, "__name__", "") == "guardia_instalaciones"]
            for t in previos:                # idempotente: no acumula guardias
                ip.input_transformers_cleanup.remove(t)
            ip.input_transformers_cleanup.append(guardia_instalaciones)
            activado["instalaciones"] = "bloqueadas"
    return activado

In [ ]:
from pathlib import Path

LIMITE_MINUTOS = 10           # sustituye por el contrato de la tarea
SIMULAR_COMPETENCIA = False  # activar después de preparar y comprobar activos
ACTIVOS_REQUERIDOS = []      # rutas a datos y pesos que TU solución realmente usa

faltantes = [str(p) for p in ACTIVOS_REQUERIDOS if not Path(p).exists()]
if faltantes:
    raise FileNotFoundError("Prepara estos activos antes del ensayo: " + ", ".join(faltantes))
if SIMULAR_COMPETENCIA:
    print(modo_competencia())
else:
    print("Aprendizaje: prepara el contrato, los datos y los modelos antes del ensayo offline.")
if not ACTIVOS_REQUERIDOS:
    print("La lista de activos está vacía: complétala al implementar tu solución.")

## 1. Semillas y dispositivo

Sin semilla fija, dos corridas dan números distintos y no sabes si tu mejora fue
real o fue suerte. Y el `device` con su `if` para que el mismo notebook corra en
tu máquina sin GPU y en la de la competencia con una.

In [ ]:
import random

import numpy as np
import torch

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
torch.cuda.manual_seed_all(SEMILLA)

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dispositivo)
if dispositivo == "cuda":
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Los datos — mirar antes de modelar

El ritual de apertura: tamaño, tipos, nulos, balance de clases, y **de dónde
viene cada fila** (si varias comparten origen, el split va por grupo).

Lee únicamente las etiquetas autorizadas por el enunciado. La mayoría de tareas
supervisadas reserva las del test, pero existen excepciones explícitas como
Double Agent Dilemma. `solo_de_train()` es una guarda opcional para cargas de
entrenamiento; comprobar el nombre de una ruta no demuestra ausencia de fuga.

In [ ]:
# datos = json.load(open(solo_de_train("dataset/train/answers.json")))
# print(len(datos))

## 3. Baseline

El primer envío. No busca puntaje: busca **verificar que el formato de salida es
correcto** y dejar anotado tu cero.

## 4. Modelo

Mide el coste completo, incluida carga de activos, entrenamiento, inferencia y
escritura. Reservar un 40% para otras fases es una heurística inicial, no una
regla del juez. Este cronómetro informa al salir del bloque; no interrumpe
un entrenamiento que se excede. En GPU sincroniza antes de medir cada fase.

In [ ]:
with Presupuesto(minutos=LIMITE_MINUTOS, nombre="entrenamiento") as p:
    pass  # entrenar acá
    print("margen: %.0f s" % p.restante())

## 5. Evaluación y salida

Con la **misma métrica** de la competencia, sobre tu validación. Y el archivo de
salida con el nombre, la forma y las claves exactas que pide el enunciado.

## 6. Ensayo de entrega

Reinicia el kernel y ejecuta todo con los activos permitidos. Comprueba que
aparecen los archivos que exige el enunciado, con sus nombres, tamaños y formas.
La ejecución completa debe caber en el límite, en el hardware correspondiente.
Una métrica local y una estimación del score oficial son resultados distintos.